In [ ]:
import os
import scanpy as sc
import numpy as np
import scvi
from scvi.model import CondSCVI, DestVI

celltype_key = "celltype_final"

for dataset_number in range(25, 26):
    sc_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad"
    st_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad"
    output_dir = f"/home/maweicheng/ST_data/SimualtedSpatalData/evaluate/Data{dataset_number}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"=== Data {dataset_number} ===")
    print(">>> Loading data")
    sc_adata = sc.read_h5ad(sc_file)
    st_adata = sc.read_h5ad(st_file)
    
    # ----------------- 1. 单细胞预处理（完全照官方逻辑） -----------------
    print(">>> Preprocessing scRNA")
    G = 2000
    
    sc.pp.filter_genes(sc_adata, min_counts=10)
    sc_adata.layers["counts"] = sc_adata.X.copy()

    sc.pp.highly_variable_genes(
        sc_adata,
        n_top_genes=G,
        subset=True,
        layer="counts",
        flavor="seurat_v3",
    )

    sc.pp.normalize_total(sc_adata, target_sum=1e5)
    sc.pp.log1p(sc_adata)

    sc_adata.raw = sc_adata
    
    # ----------------- 2. 空间数据预处理 -----------------
    print(">>> Preprocessing spatial")
    st_adata.layers["counts"] = st_adata.X.copy()
    sc.pp.normalize_total(st_adata, target_sum=1e5)
    sc.pp.log1p(st_adata)
    st_adata.raw = st_adata

    # 基因交集（保证模型两边基因一致）
    intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
    sc_adata = sc_adata[:, intersect].copy()
    st_adata = st_adata[:, intersect].copy()

    # 可选：过滤掉极低 UMI 的 spot
    # st_adata = st_adata[st_adata.layers["counts"].sum(1) > 10].copy()
    
    # ----------------- 3. 训练 CondSCVI（scLVM） -----------------
    print(">>> Training CondSCVI")
    
    CondSCVI.setup_anndata(sc_adata, layer="counts", labels_key=celltype_key)
          # 或 True，看你是否要平衡 cell type
    sc_model = CondSCVI(sc_adata, weight_obs=False)
    sc_model.train(max_epochs=250, lr=1e-4)

    # 如果需要，可以保存模型
    # sc_model.save(os.path.join(output_dir, "CondSCVI_model"), overwrite=True)

    # ----------------- 4. 训练 DestVI（stLVM） -----------------
    print(">>> Training DestVI")
    # 可选：如果有空间坐标，做 smoothing 并传 smoothed_layer；这里先用最简单版本
    DestVI.setup_anndata(st_adata, layer="counts")
    st_model = DestVI.from_rna_model(st_adata, sc_model)
    st_model.train(max_epochs=2500)

    # ----------------- 5. 导出 proportion -----------------
    print(">>> Saving results")
    proportions = st_model.get_proportions()  # DataFrame
    
    proportions.to_csv(os.path.join(output_dir, "DestVI.csv"))

In [8]:
import os
import scanpy as sc
import numpy as np
import scvi
from scvi.model import CondSCVI, DestVI

celltype_key = "celltype"

sc_file = f"/home/maweicheng/ST_data/STARmap/starmap_sc_rna.h5ad"
st_file = f"/home/maweicheng/ST_data/STARmap/STARmap_SP.h5ad"
output_dir = f"/home/maweicheng/ST_data/STARmap"
os.makedirs(output_dir, exist_ok=True)

print(">>> Loading data")
sc_adata = sc.read_h5ad(sc_file)
st_adata = sc.read_h5ad(st_file)

# ----------------- 1. 单细胞预处理（完全照官方逻辑） -----------------
print(">>> Preprocessing scRNA")
G = 2000

sc.pp.filter_genes(sc_adata, min_counts=10)
sc_adata.layers["counts"] = sc_adata.X.copy()

sc.pp.highly_variable_genes(
    sc_adata,
    n_top_genes=G,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
)

sc.pp.normalize_total(sc_adata, target_sum=1e5)
sc.pp.log1p(sc_adata)
sc_adata.raw = sc_adata

# ----------------- 2. 空间数据预处理 -----------------
print(">>> Preprocessing spatial")
st_adata.layers["counts"] = st_adata.X.copy()
sc.pp.normalize_total(st_adata, target_sum=1e5)
sc.pp.log1p(st_adata)
st_adata.raw = st_adata

# 基因交集（保证模型两边基因一致）
intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
sc_adata = sc_adata[:, intersect].copy()
st_adata = st_adata[:, intersect].copy()

# 可选：过滤掉极低 UMI 的 spot
# st_adata = st_adata[st_adata.layers["counts"].sum(1) > 10].copy()

# ----------------- 3. 训练 CondSCVI（scLVM） -----------------
print(">>> Training CondSCVI")

CondSCVI.setup_anndata(sc_adata, layer="counts", labels_key=celltype_key)
        # 或 True，看你是否要平衡 cell type
sc_model = CondSCVI(sc_adata, weight_obs=False)
sc_model.train(max_epochs=250, lr=1e-4)

# 如果需要，可以保存模型
# sc_model.save(os.path.join(output_dir, "CondSCVI_model"), overwrite=True)

# ----------------- 4. 训练 DestVI（stLVM） -----------------
print(">>> Training DestVI")
# 可选：如果有空间坐标，做 smoothing 并传 smoothed_layer；这里先用最简单版本
DestVI.setup_anndata(st_adata, layer="counts")
st_model = DestVI.from_rna_model(st_adata, sc_model)
st_model.train(max_epochs=2500)

# ----------------- 5. 导出 proportion -----------------
print(">>> Saving results")
proportions = st_model.get_proportions()  # DataFrame

proportions.to_csv(os.path.join(output_dir, "DestVI.csv"))

>>> Loading data
>>> Preprocessing scRNA


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=111` in the `DataLoader` to improve performance.


>>> Preprocessing spatial
>>> Training CondSCVI
Epoch 250/250: 100%|██████████| 250/250 [06:08<00:00,  1.58s/it, v_num=1, train_loss_step=3.36e+3, train_loss_epoch=3.15e+3]

`Trainer.fit` stopped: `max_epochs=250` reached.


Epoch 250/250: 100%|██████████| 250/250 [06:08<00:00,  1.47s/it, v_num=1, train_loss_step=3.36e+3, train_loss_epoch=3.15e+3]
>>> Training DestVI


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=111` in the `DataLoader` to improve performance.
/softwares/miniconda3/envs/cell2loc_env/lib/python3.9/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want t

Epoch 2500/2500: 100%|██████████| 2500/2500 [01:46<00:00, 23.91it/s, v_num=1, train_loss_step=7.54e+4, train_loss_epoch=7.67e+4]

`Trainer.fit` stopped: `max_epochs=2500` reached.


Epoch 2500/2500: 100%|██████████| 2500/2500 [01:46<00:00, 23.44it/s, v_num=1, train_loss_step=7.54e+4, train_loss_epoch=7.67e+4]
>>> Saving results
